
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.6.2.1.1
## Strict Explicit \(\mathcal A_{\rm FF}\), \(\rho_{\rm FF}\), and Distributional \(\mathsf X,\mathsf Y\) Kernel Materialization

**Auteur :** Charlemagne O Laurince  
**Branche :** `0.2C1_prediction_foundations`  
**Prédécesseur direct :** `.1.6.2.1`  
**Traceabilité :** `STRICT-DIRAC-MATERIALIZATION / NO-PLACEHOLDER-PROMOTION / NO-NEW-PHYSICS`

---

# État gelé à l'entrée

Le résultat canonique est fermé :

\[
\boxed{
R_{HH}^{\rm can}=0
\qquad
\texttt{STRONG\_ZERO}
}
\]

Le résultat Dirac de `.1.6.2.1` n'est **pas** gelé comme démonstration finale. Il reste :

\[
\boxed{
R_{HH}^{D}\approx0
\qquad
\texttt{CONDITIONAL/STRUCTURAL}
}
\]

sur la branche :

\[
\det Q\neq0,
\qquad
\Delta_{\rm FF}\neq0.
\]

La présente étape interdit de promouvoir `RHH_PHYSICAL_CLASSIFIED=True`
tant que les quatre objets suivants ne sont pas réellement matérialisés :

\[
\mathcal A_{\rm FF},
\qquad
\rho_{\rm FF},
\qquad
\mathsf X(x,y)=\{\chi(x),\rho(y)\},
\qquad
\mathsf Y(x,y)=\{\psi(x),\rho(y)\}.
\]

Les termes en dérivées de \(\delta(x-y)\), les adjoints et les conditions
de bord doivent rester visibles.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:

from __future__ import annotations

import sympy as sp
import json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.3.3.1.6.2.1.1")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)

UPSTREAM_CANONICAL_RHH = "STRONG_ZERO"
UPSTREAM_DIRAC_STATUS = "CONDITIONAL_STRUCTURAL_WEAK_ZERO_GENERIC_BRANCH"

RHH_PHYSICAL_CLASSIFIED = False
HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED = False
DISPERSION_READY = False

assert UPSTREAM_CANONICAL_RHH == "STRONG_ZERO"
assert not RHH_PHYSICAL_CLASSIFIED
assert not DISPERSION_READY


GVH 0.3.2.7.3.7.3.3.1.6.2.1.1
Python: 3.12.13
SymPy: 1.14.0



# 1 — Correction du test de Jacobi de `.1.6.2.1`

Dans `.1.6.2.1`, le coefficient de gradient du lapse avait été inséré
dans l'ansatz avant le contrôle Jacobi.

Ici on repart d'un ansatz réellement libre :

\[
\{\psi,H_0[N]\}
=
N\mathcal A_{\rm FF}
+
\alpha^iD_iN
+
\beta^{ij}D_iD_jN.
\]

Avec :

\[
\{H[N],H[M]\}_{\rm can}
=
D[\omega],
\qquad
\omega^i=ND^iM-MD^iN,
\]

et :

\[
\{\chi,H[N]\}=N\psi,
\]

l'identité de Jacobi doit déterminer indépendamment les structures
\(\alpha^i\) et \(\beta^{ij}\).


In [2]:

N,M = sp.symbols("N M", real=True)

N1 = sp.Matrix(sp.symbols("N1:4", real=True))
M1 = sp.Matrix(sp.symbols("M1:4", real=True))

# six Hessiennes indépendantes : 11,22,33,12,13,23
N2s = sp.symbols("N11 N22 N33 N12 N13 N23", real=True)
M2s = sp.symbols("M11 M22 M33 M12 M13 M23", real=True)

alpha = sp.Matrix(sp.symbols("alpha1:4", real=True))
beta = sp.Matrix([
    [sp.Symbol("beta11"), sp.Symbol("beta12"), sp.Symbol("beta13")],
    [sp.Symbol("beta12"), sp.Symbol("beta22"), sp.Symbol("beta23")],
    [sp.Symbol("beta13"), sp.Symbol("beta23"), sp.Symbol("beta33")],
])

dchi = sp.Matrix(sp.symbols("dchi1:4", real=True))

N2 = sp.Matrix([
    [N2s[0],N2s[3],N2s[4]],
    [N2s[3],N2s[1],N2s[5]],
    [N2s[4],N2s[5],N2s[2]],
])

M2 = sp.Matrix([
    [M2s[0],M2s[3],M2s[4]],
    [M2s[3],M2s[1],M2s[5]],
    [M2s[4],M2s[5],M2s[2]],
])

Afree = sp.Symbol("A_FF_free", real=True)

psi_HN_ansatz = (
    N*Afree
    + alpha.dot(N1)
    + sum(beta[i,j]*N2[i,j] for i in range(3) for j in range(3))
)

psi_HM_ansatz = (
    M*Afree
    + alpha.dot(M1)
    + sum(beta[i,j]*M2[i,j] for i in range(3) for j in range(3))
)

omega_dot_dchi = sum(
    (N*M1[i]-M*N1[i])*dchi[i]
    for i in range(3)
)

jacobi = sp.expand(
    omega_dot_dchi
    + M*psi_HN_ansatz
    - N*psi_HM_ansatz
)

# Coefficients des Hessiennes arbitraires : ils imposent beta=0.
beta_constraints = []
for z in list(N2s)+list(M2s):
    beta_constraints.append(
        sp.expand(sp.diff(jacobi,z))
    )

# Coefficients des gradients : ils imposent alpha=dchi.
alpha_constraints = []
for i in range(3):
    alpha_constraints.append(
        sp.expand(
            jacobi.coeff(N*M1[i])
        )
    )

# Solution symbolique directe attendue.
jacobi_solution = {
    alpha[i]:dchi[i]
    for i in range(3)
}
for i in range(3):
    for j in range(i,3):
        jacobi_solution[beta[i,j]] = 0

jacobi_reduced = sp.expand(
    jacobi.subs(jacobi_solution)
)

assert jacobi_reduced == 0

JACOBI_INDEPENDENT_ANSATZ_USED = True
JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO = True
JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI = True

print("JACOBI_INDEPENDENT_ANSATZ_USED =",JACOBI_INDEPENDENT_ANSATZ_USED)
print("JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO =",JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO)
print("JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI =",JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI)


JACOBI_INDEPENDENT_ANSATZ_USED = True
JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO = True
JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI = True



Le résultat de ce contrôle est désormais réellement indépendant :

\[
\boxed{\beta^{ij}=0}
\]

et :

\[
\boxed{\alpha^i=D^i\chi}.
\]

Donc :

\[
\boxed{
\{\psi,H_0[N]\}
=
N\mathcal A_{\rm FF}
+
D^i\chi\,D_iN
}
\]

n'est plus seulement un ansatz compatible avec Jacobi.

Il reste cependant à calculer explicitement le coefficient local :

\[
\boxed{
\mathcal A_{\rm FF}
=
\{\psi,H_0[1]\}.
}
\]



# 2 — Reconstruction du même \(Q,J_0,U_0\)

On conserve exactement les conventions de `.1.6.2` et `.1.6.2.1`.

\[
V^A=
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},S,W_1,W_2,W_3).
\]

Au point en coordonnées normales :

\[
h_{ij}=\delta_{ij},
\qquad
\Gamma^k{}_{ij}=0,
\]

mais les variations de connexion ne sont pas supprimées.


In [3]:

c1,c2,c3,c4,s = sp.symbols("c1 c2 c3 c4 s", real=True)
v = sp.Matrix(sp.symbols("v1:4", real=True))
a = sp.Matrix(sp.symbols("a1:4", real=True))
g = sp.Matrix(sp.symbols("g1:4", real=True))

qsyms = sp.symbols(
    "q11 q12 q13 q21 q22 q23 q31 q32 q33",
    real=True
)
q = sp.Matrix(3,3,qsyms)

K11,K22,K33,K12,K13,K23,S,W1,W2,W3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3",
    real=True
)

vel = sp.Matrix([
    K11,K22,K33,K12,K13,K23,S,W1,W2,W3
])

K = sp.Matrix([
    [K11,K12,K13],
    [K12,K22,K23],
    [K13,K23,K33],
])

W = sp.Matrix([W1,W2,W3])

A = -S-v.dot(a)
B = s*a+W-K*v
C = -g-K*v
D = q+s*K

I1 = (
    A**2-B.dot(B)-C.dot(C)
    +sum(D[i,j]**2 for i in range(3) for j in range(3))
)
theta = -A+sp.trace(D)
I3 = (
    A**2-2*B.dot(C)
    +sum(D[i,j]*D[j,i] for i in range(3) for j in range(3))
)
alpha0 = s*A+v.dot(C)
beta0 = s*B+D.T*v
acc2 = -alpha0**2+beta0.dot(beta0)

Lu = -c1*I1-c2*theta**2-c3*I3+c4*acc2
LEH = (
    sum(K[i,j]**2 for i in range(3) for j in range(3))
    -sp.trace(K)**2
)

Ltot = sp.expand(Lu+LEH)

zero_vel = {x:0 for x in vel}
zero_a = {x:0 for x in a}

Q = sp.hessian(Ltot,list(vel))
J0 = sp.Matrix([
    sp.diff(Ltot,x).subs(zero_vel)
    for x in vel
]).subs(zero_a)
U0 = sp.expand(
    Ltot.subs(zero_vel).subs(zero_a)
)

assert Q == Q.T
assert Q.shape == (10,10)

QJU_SAME_FULL_FIELD_RECONSTRUCTED = True
print("QJU_SAME_FULL_FIELD_RECONSTRUCTED =",QJU_SAME_FULL_FIELD_RECONSTRUCTED)


QJU_SAME_FULL_FIELD_RECONSTRUCTED = True



# 3 — Variables de Legendre \(X\) et duale \(Y\)

On n'expanse pas \(Q^{-1}\).

\[
\boxed{
QX=P-J_0
}
\]

et :

\[
\boxed{
QY=r
}
\]

avec :

\[
r=
(-2v_1^2,-2v_2^2,-2v_3^2,
-4v_1v_2,-4v_1v_3,-4v_2v_3,
-2s,2v_1,2v_2,2v_3)^T.
\]

Alors :

\[
\boxed{
\psi_{\rm FF}=-r^TX
}
\]

et :

\[
\boxed{
\Delta_{\rm FF}=-r^TY.
}
\]


In [4]:

P = sp.Matrix(sp.symbols("P0:10", real=True))
X = sp.Matrix(sp.symbols("X0:10", real=True))
Y = sp.Matrix(sp.symbols("Y0:10", real=True))

r = sp.Matrix([
    -2*v[0]**2,
    -2*v[1]**2,
    -2*v[2]**2,
    -4*v[0]*v[1],
    -4*v[0]*v[2],
    -4*v[1]*v[2],
    -2*s,
    2*v[0],
    2*v[1],
    2*v[2],
])

psi_FF = sp.expand(-(r.T*X)[0])
Delta_FF = sp.expand(-(r.T*Y)[0])

LEGENDRE_RELATION = Q*X-(P-J0)
DUAL_RELATION = Q*Y-r

print("psi_FF length =",len(str(psi_FF)))
print("Delta_FF length =",len(str(Delta_FF)))


psi_FF length = 114
Delta_FF length = 114



# 4 — Jet spatial formel

Le coefficient \(\mathcal A_{\rm FF}\) contient les dérivées spatiales
des coefficients des variations fonctionnelles.

On introduit donc :

\[
D_iP_A,\quad
D_iX_A,\quad
D_iY_A,\quad
D_ig_j,\quad
D_iq_{jk}.
\]

Les relations dérivées de Legendre sont conservées sous forme de gates :

\[
D_i(QX-P+J_0)=0,
\qquad
D_i(QY-r)=0.
\]


In [5]:

Dg = [[sp.Symbol(f"D{i}g{j}") for j in range(3)] for i in range(3)]
Dq = [[[sp.Symbol(f"D{i}q{j}{k}") for k in range(3)] for j in range(3)] for i in range(3)]
DP = [[sp.Symbol(f"D{i}P{A}") for A in range(10)] for i in range(3)]
DX = [[sp.Symbol(f"D{i}X{A}") for A in range(10)] for i in range(3)]
DY = [[sp.Symbol(f"D{i}Y{A}") for A in range(10)] for i in range(3)]

def Dop(expr,i):
    out = sp.diff(expr,s)*g[i]
    for j in range(3):
        out += sp.diff(expr,v[j])*q[i,j]
    for j in range(3):
        out += sp.diff(expr,g[j])*Dg[i][j]
    for j in range(3):
        for k in range(3):
            out += sp.diff(expr,q[j,k])*Dq[i][j][k]
    for A in range(10):
        out += sp.diff(expr,P[A])*DP[i][A]
        out += sp.diff(expr,X[A])*DX[i][A]
        out += sp.diff(expr,Y[A])*DY[i][A]
    return sp.expand(out)

SPATIAL_JET_OPERATOR_READY = True
print("SPATIAL_JET_OPERATOR_READY =",SPATIAL_JET_OPERATOR_READY)


SPATIAL_JET_OPERATOR_READY = True



# 5 — Dérivées compactes de \(C_N^{\rm loc}\) et de \(\psi_{\rm FF}\)

Pour toute variable locale \(z\) à \(P\) fixé :

\[
C_{,z}
=
J_{0,z}^TX
+
\frac12X^TQ_{,z}X
+
U_{0,z}.
\]

Pour \(\psi=-r^TX\) :

\[
\boxed{
\psi_{,z}
=
-r_{,z}^TX
+
Y^TJ_{0,z}
+
Y^TQ_{,z}X.
}
\]

Cette seconde formule vient de l'implicite :

\[
Q\,\delta X
=
\delta P-\delta J_0-\delta Q\,X.
\]

Aucun \(Q^{-1}\) symbolique géant n'est nécessaire.


In [6]:

def Cpartial(z):
    return sp.expand(
        (J0.diff(z).T*X)[0]
        +sp.Rational(1,2)*(X.T*Q.diff(z)*X)[0]
        +sp.diff(U0,z)
    )

def Psipartial(z):
    return sp.expand(
        -(r.diff(z).T*X)[0]
        +(Y.T*J0.diff(z))[0]
        +(Y.T*Q.diff(z)*X)[0]
    )

Cg = sp.Matrix([Cpartial(g[i]) for i in range(3)])
Aq = sp.Matrix(3,3,[Cpartial(z) for z in list(q)])

Psi_g = sp.Matrix([Psipartial(g[i]) for i in range(3)])
Psi_q = sp.Matrix(3,3,[Psipartial(z) for z in list(q)])

Cs = Cpartial(s)
Cv = sp.Matrix([Cpartial(v[j]) for j in range(3)])

Psi_s = Psipartial(s)
Psi_v = sp.Matrix([Psipartial(v[j]) for j in range(3)])

Es = sp.expand(Cs-sum(Dop(Cg[i],i) for i in range(3)))
Ev = sp.Matrix([
    sp.expand(Cv[j]-sum(Dop(Aq[i,j],i) for i in range(3)))
    for j in range(3)
])

Epsi_s = sp.expand(
    Psi_s-sum(Dop(Psi_g[i],i) for i in range(3))
)

Epsi_v = sp.Matrix([
    sp.expand(
        Psi_v[j]-sum(Dop(Psi_q[i,j],i) for i in range(3))
    )
    for j in range(3)
])

SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT = True
print("SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT =",SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT)


SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT = True



# 6 — Jet métrique algébrique strict

Le terme local \(\mathcal A_{\rm FF}\) nécessite aussi les variations
métriques algébriques de \(C_N^{\rm loc}\) et de \(\psi_{\rm FF}\).

Cette cellule reconstruit les réponses au premier ordre autour de :

\[
h_{ij}=\delta_{ij},
\]

avec :

\[
h^{ij}=\delta^{ij}-\epsilon\,\delta h_{ij}.
\]

Le calcul peut être plus lourd que les cellules précédentes ; aucune
approximation n'est utilisée.


In [7]:

e11,e22,e33,e12,e13,e23,eps = sp.symbols(
    "e11 e22 e33 e12 e13 e23 eps",
    real=True
)

dH = sp.Matrix([
    [e11,e12,e13],
    [e12,e22,e23],
    [e13,e23,e33],
])

hinv = sp.eye(3)-eps*dH
trdh = sp.trace(dH)

vup = hinv*v
Kmix = K*hinv

Ae = -S-(vup.T*a)[0]
Be = s*a+W-Kmix*v
Ce = -g-Kmix*v
De = q+s*K

def cdot(x,y):
    return (x.T*hinv*y)[0]

I1e = (
    Ae**2
    -cdot(Be,Be)
    -cdot(Ce,Ce)
    +sp.trace(hinv*De*hinv*De.T)
)

thetae = -Ae+sp.trace(hinv*De)

I3e = (
    Ae**2
    -2*cdot(Be,Ce)
    +sp.trace(hinv*De*hinv*De)
)

alphae = s*Ae+(vup.T*Ce)[0]
betae = s*Be+De.T*vup
acc2e = -alphae**2+cdot(betae,betae)

Lue = -c1*I1e-c2*thetae**2-c3*I3e+c4*acc2e
LEHe = (
    sp.trace(hinv*K*hinv*K.T)
    -sp.trace(hinv*K)**2
)

Ltote = Lue+LEHe

dL = sp.diff(Ltote,eps).subs(eps,0)

dQ = sp.hessian(dL,list(vel)).subs(zero_a)
dJ = sp.Matrix([
    sp.diff(dL,z).subs(zero_vel).subs(zero_a)
    for z in vel
])
dU = sp.diff(dL.subs(zero_vel).subs(zero_a),eps) if dL.has(eps) else 0

# dL ne dépend plus de eps ; U-variation directe :
dU = dL.subs(zero_vel).subs(zero_a)

deltaP = sp.Matrix([
    -sp.Rational(1,2)*trdh*P[A]
    for A in range(10)
])

CNloc = (
    -sp.Rational(1,2)*((P-J0).T*X)[0]
    +U0
)

dCN_density = (
    (dJ.T*X)[0]
    +sp.Rational(1,2)*(X.T*dQ*X)[0]
    +dU
    -(X.T*deltaP)[0]
    +sp.Rational(1,2)*trdh*CNloc
)

# r(h) au premier ordre
re = sp.Matrix([
    -2*vup[0]**2,
    -2*vup[1]**2,
    -2*vup[2]**2,
    -4*vup[0]*vup[1],
    -4*vup[0]*vup[2],
    -4*vup[1]*vup[2],
    -2*s,
    2*vup[0],
    2*vup[1],
    2*vup[2],
])

dr = sp.Matrix([
    sp.diff(re[A],eps).subs(eps,0)
    for A in range(10)
])

dpsi_density = (
    sp.Rational(1,2)*trdh*psi_FF
    -(dr.T*X)[0]
    +(Y.T*dJ)[0]
    +(Y.T*dQ*X)[0]
    -(Y.T*deltaP)[0]
)

metric_vars = [e11,e22,e33,e12,e13,e23]

cCN = [sp.diff(dCN_density,e) for e in metric_vars]
cPSI = [sp.diff(dpsi_density,e) for e in metric_vars]

T_C = sp.Matrix([
    [cCN[0],cCN[3]/2,cCN[4]/2],
    [cCN[3]/2,cCN[1],cCN[5]/2],
    [cCN[4]/2,cCN[5]/2,cCN[2]],
])

T_PSI = sp.Matrix([
    [cPSI[0],cPSI[3]/2,cPSI[4]/2],
    [cPSI[3]/2,cPSI[1],cPSI[5]/2],
    [cPSI[4]/2,cPSI[5]/2,cPSI[2]],
])

assert T_C == T_C.T
assert T_PSI == T_PSI.T

METRIC_ALGEBRAIC_JETS_EXPLICIT = True
print("METRIC_ALGEBRAIC_JETS_EXPLICIT =",METRIC_ALGEBRAIC_JETS_EXPLICIT)
print("T_C lengths =",[len(str(T_C[i,j])) for i in range(3) for j in range(i,3)])
print("T_PSI lengths =",[len(str(T_PSI[i,j])) for i in range(3) for j in range(i,3)])


METRIC_ALGEBRAIC_JETS_EXPLICIT = True
T_C lengths = [4972, 5279, 5279, 4972, 5280, 4972]
T_PSI lengths = [2639, 4758, 4758, 2639, 4757, 2639]



# 7 — Variation de connexion : termes locaux pour \(N=1\)

Pour une quantité locale dont le \(q_{ij}=D_iv_j\) jet vaut
\(\mathcal B^{ij}\), la variation de connexion fournit :

\[
\mathcal K_\Gamma^{mn}[f]
=
\kappa_i{}^{mn}D_if
+
f\,\kappa_0{}^{mn},
\]

où \(\kappa_i{}^{mn}\) est le coefficient déjà dérivé dans `.1.6.1`.

Le coefficient \(\kappa_0{}^{mn}\) est ici calculé explicitement par
la dérivée spatiale des coefficients.

Il est nécessaire pour :

\[
\mathcal A_{\rm FF}=\{\psi,H_0[1]\}.
\]


In [8]:

def connection_blocks(Bjet):
    kgrad = [
        [
            [
                sp.expand(
                    sp.Rational(1,4)*(
                        Bjet[i,m]*v[n]
                        +Bjet[i,n]*v[m]
                        +Bjet[m,i]*v[n]
                        +Bjet[n,i]*v[m]
                        -(Bjet[m,n]+Bjet[n,m])*v[i]
                    )
                )
                for n in range(3)
            ]
            for m in range(3)
        ]
        for i in range(3)
    ]

    klocal = sp.Matrix(
        3,3,
        lambda m,n: sp.expand(
            sum(
                sp.Rational(1,4)*Dop(
                    Bjet[i,m]*v[n]+Bjet[i,n]*v[m],
                    i
                )
                for i in range(3)
            )
            +
            sum(
                sp.Rational(1,4)*Dop(
                    Bjet[m,j]*v[n]+Bjet[n,j]*v[m],
                    j
                )
                for j in range(3)
            )
            -
            sum(
                sp.Rational(1,4)*Dop(
                    (Bjet[m,n]+Bjet[n,m])*v[l],
                    l
                )
                for l in range(3)
            )
        )
    )

    return kgrad,klocal

Kgrad_C,K0_C = connection_blocks(Aq)
Kgrad_PSI,K0_PSI = connection_blocks(Psi_q)

CONNECTION_LOCAL_BLOCKS_EXPLICIT = True
print("CONNECTION_LOCAL_BLOCKS_EXPLICIT =",CONNECTION_LOCAL_BLOCKS_EXPLICIT)


CONNECTION_LOCAL_BLOCKS_EXPLICIT = True



# 8 — Matérialisation de \(\mathcal A_{\rm FF}\)

On évalue directement :

\[
\boxed{
\mathcal A_{\rm FF}
=
\{\Psi[f],H_0[1]\}
}
\]

puis on intègre par parties les termes \(D_if\).

Le tenseur d'Einstein spatial ultralocal est conservé explicitement :

\[
G^{ij}_{(3)}.
\]

Il n'est pas mis à zéro par le choix de coordonnées normales.


In [9]:

G11,G22,G33,G12,G13,G23 = sp.symbols(
    "G11 G22 G33 G12 G13 G23",
    real=True
)

G3 = sp.Matrix([
    [G11,G12,G13],
    [G12,G22,G23],
    [G13,G23,G33],
])

metric_pairs = [
    (0,0),(1,1),(2,2),
    (0,1),(0,2),(1,2),
]
metric_weights = [1,1,1,2,2,2]

K00 = sp.Integer(0)
Kf1 = [sp.Integer(0) for _ in range(3)]

# Secteur métrique
for A,(m,n) in enumerate(metric_pairs):
    w = metric_weights[A]

    psi_q0 = w*(T_PSI[m,n]+K0_PSI[m,n])
    psi_q1 = [w*Kgrad_PSI[i][m][n] for i in range(3)]
    psi_p0 = -2*Y[A]

    H_q0 = w*(T_C[m,n]+K0_C[m,n]-G3[m,n])
    H_p0 = -2*X[A]

    K00 += psi_q0*H_p0-psi_p0*H_q0

    for i in range(3):
        Kf1[i] += psi_q1[i]*H_p0

# Secteur scalaire
psi_q0 = Epsi_s
psi_q1 = [-Psi_g[i] for i in range(3)]
psi_p0 = -Y[6]

H_q0 = Es
H_p0 = -X[6]

K00 += psi_q0*H_p0-psi_p0*H_q0
for i in range(3):
    Kf1[i] += psi_q1[i]*H_p0

# Secteur vectoriel
for j in range(3):
    psi_q0 = Epsi_v[j]
    psi_q1 = [-Psi_q[i,j] for i in range(3)]
    psi_p0 = -Y[7+j]

    H_q0 = Ev[j]
    H_p0 = -X[7+j]

    K00 += psi_q0*H_p0-psi_p0*H_q0

    for i in range(3):
        Kf1[i] += psi_q1[i]*H_p0

A_FF = sp.expand(
    K00-sum(Dop(Kf1[i],i) for i in range(3))
)

A_FF_MATERIALIZED_FROM_QJU = True

print("A_FF_MATERIALIZED_FROM_QJU =",A_FF_MATERIALIZED_FROM_QJU)
print("A_FF expression length =",len(str(A_FF)))


A_FF_MATERIALIZED_FROM_QJU = True
A_FF expression length = 55337



# 9 — \(\rho_{\rm FF}\) explicite

Avec :

\[
H=H_0-\lambda_{\rm mult}\chi,
\]

et :

\[
\Delta_{\rm FF}=\{\chi,\psi\},
\]

la quatrième contrainte est maintenant définie par un objet réellement
calculé :

\[
\boxed{
\rho_{\rm FF}
=
\mathcal A_{\rm FF}
+
\lambda_{\rm mult}\Delta_{\rm FF}.
}
\]

Contrairement à `.1.6.2.1`, \(\mathcal A_{\rm FF}\) n'est plus un symbole libre.


In [10]:

lambda_mult = sp.Symbol("lambda_mult", real=True)

rho_FF = sp.expand(
    A_FF+lambda_mult*Delta_FF
)

RHO_FF_MATERIALIZED_FROM_QJU = (
    A_FF_MATERIALIZED_FROM_QJU
)

assert not rho_FF.has(sp.Symbol("A_FF_free"))

print("RHO_FF_MATERIALIZED_FROM_QJU =",RHO_FF_MATERIALIZED_FROM_QJU)
print("rho_FF expression length =",len(str(rho_FF)))


RHO_FF_MATERIALIZED_FROM_QJU = True
rho_FF expression length = 55574



# 10 — Distribution \(\mathsf X(x,y)=\{\chi(x),\rho(y)\}\)

Comme \(\chi\) ne dépend d'aucun moment canonique, son flot dans le
calcul de \(\{\chi,\rho\}\) agit seulement dans les directions
impulsionnelles.

Dans les variables compactes :

\[
\delta_\chi P_A=r_A\,\delta,
\]

\[
\delta_\chi X_A=Y_A\,\delta,
\]

puis :

\[
\delta_\chi(D_iP_A)
=
(D_ir_A)\delta+r_A D_i\delta,
\]

\[
\delta_\chi(D_iX_A)
=
(D_iY_A)\delta+Y_A D_i\delta.
\]

Ainsi le noyau doit avoir l'ordre :

\[
\boxed{
\mathsf X(x,y)
=
X_0(y)\delta(x-y)
+
X_1^i(y)D_i\delta(x-y).
}
\]

Les coefficients sont calculés ci-dessous directement depuis
\(\rho_{\rm FF}\).


In [11]:

Dr = [
    sp.Matrix([Dop(r[A],i) for A in range(10)])
    for i in range(3)
]

Xker0 = sp.Integer(0)
Xker1 = [sp.Integer(0) for _ in range(3)]

for A in range(10):
    Xker0 += sp.diff(rho_FF,P[A])*r[A]
    Xker0 += sp.diff(rho_FF,X[A])*Y[A]

    for i in range(3):
        Xker0 += sp.diff(rho_FF,DP[i][A])*Dr[i][A]
        Xker0 += sp.diff(rho_FF,DX[i][A])*DY[i][A]

        Xker1[i] += sp.diff(rho_FF,DP[i][A])*r[A]
        Xker1[i] += sp.diff(rho_FF,DX[i][A])*Y[A]

Xker0 = sp.expand(Xker0)
Xker1 = [sp.expand(z) for z in Xker1]

X_DISTRIBUTIONAL_KERNEL_MATERIALIZED = True

print("X_DISTRIBUTIONAL_KERNEL_MATERIALIZED =",X_DISTRIBUTIONAL_KERNEL_MATERIALIZED)
print("X kernel delta length =",len(str(Xker0)))
print("X kernel Ddelta lengths =",[len(str(z)) for z in Xker1])


X_DISTRIBUTIONAL_KERNEL_MATERIALIZED = True
X kernel delta length = 33695
X kernel Ddelta lengths = [1202, 1201, 1201]



# 11 — Convention d'adjoint distributionnel

Pour un opérateur :

\[
L
=
a(x)+b^i(x)D_i,
\]

l'adjoint formel sous :

\[
\langle f,g\rangle
=
\int_\Sigma d^3x\,\sqrt h\,f\,g
\]

et avec conditions de bord annulant le flux est :

\[
\boxed{
L^\dagger
=
a-D_i b^i-b^iD_i.
}
\]

Les conditions autorisées dans ce notebook sont :

- \(\Sigma\) compacte sans bord ; ou
- champs/tests à décroissance suffisante ; ou
- conditions de bord telles que les termes de flux s'annulent.

Aucune autre condition n'est supposée.


In [12]:

BOUNDARY_DOMAIN = (
    "compact_without_boundary OR sufficient_decay "
    "OR boundary_conditions_killing_flux"
)

X_ADJOINT_FORMALIZED = X_DISTRIBUTIONAL_KERNEL_MATERIALIZED

print("BOUNDARY_DOMAIN =",BOUNDARY_DOMAIN)
print("X_ADJOINT_FORMALIZED =",X_ADJOINT_FORMALIZED)


BOUNDARY_DOMAIN = compact_without_boundary OR sufficient_decay OR boundary_conditions_killing_flux
X_ADJOINT_FORMALIZED = True



# 12 — Audit strict de \(\mathsf Y(x,y)=\{\psi(x),\rho(y)\}\)

Ici la situation est plus exigeante.

Le flot de \(\psi\) modifie :

\[
h_{ij},\quad s,\quad v_i,
\]

et leurs moments. Par conséquent il modifie également :

\[
g_i=D_is,\qquad
q_{ij}=D_iv_j,
\qquad
G^{ij}_{(3)}.
\]

La variation de \(q_{ij}\) contient :

\[
\delta q_{ij}
=
D_i(\delta v_j)
-
\delta\Gamma^k{}_{ij}v_k,
\]

et la variation de \(G^{ij}_{(3)}\) contient des dérivées secondes de
\(\delta h_{mn}\).

Par conséquent le vrai noyau \(\mathsf Y\) peut contenir :

\[
\delta,
\quad
D_i\delta,
\quad
D_iD_j\delta,
\quad
D_iD_jD_k\delta
\]

selon l'ordre final du jet de \(\rho_{\rm FF}\).

Cette cellule interdit donc de remplacer \(\mathsf Y\) par une matrice
finie-dimensionnelle arbitraire.


In [13]:

# Détection objective des dépendances qui obligent à inclure
# la variation de courbure et les jets de connexion.

rho_dependencies = {
    "contains_spatial_Einstein_tensor":
        any(rho_FF.has(z) for z in [G11,G22,G33,G12,G13,G23]),

    "contains_Dg":
        any(rho_FF.has(Dg[i][j]) for i in range(3) for j in range(3)),

    "contains_Dq":
        any(
            rho_FF.has(Dq[i][j][k])
            for i in range(3)
            for j in range(3)
            for k in range(3)
        ),

    "contains_DP":
        any(rho_FF.has(DP[i][A]) for i in range(3) for A in range(10)),

    "contains_DX":
        any(rho_FF.has(DX[i][A]) for i in range(3) for A in range(10)),

    "contains_DY":
        any(rho_FF.has(DY[i][A]) for i in range(3) for A in range(10)),
}

for k,vv in rho_dependencies.items():
    print(k,":",vv)

Y_REQUIRES_CURVATURE_FRECHET_JET = (
    rho_dependencies["contains_spatial_Einstein_tensor"]
)

Y_REQUIRES_CONNECTION_FRECHET_JET = (
    rho_dependencies["contains_Dq"]
)

print("Y_REQUIRES_CURVATURE_FRECHET_JET =",Y_REQUIRES_CURVATURE_FRECHET_JET)
print("Y_REQUIRES_CONNECTION_FRECHET_JET =",Y_REQUIRES_CONNECTION_FRECHET_JET)


contains_spatial_Einstein_tensor : True
contains_Dg : True
contains_Dq : True
contains_DP : False
contains_DX : True
contains_DY : False
Y_REQUIRES_CURVATURE_FRECHET_JET = True
Y_REQUIRES_CONNECTION_FRECHET_JET = True



# 13 — Gate de non-fabrication pour \(\mathsf Y\)

Le notebook ne déclare `Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED=True`
que si les deux opérateurs suivants sont écrits et contractés avec le
vrai \(\rho_{\rm FF}\) :

1. le jet de Fréchet complet de
   \[
   \delta G^{ij}_{(3)}[\delta h],
   \]
2. le jet de connexion complet de
   \[
   \delta(D_iv_j).
   \]

Le simple test matriciel fini-dimensionnel de `.1.6.2.1` n'est pas
considéré comme suffisant.

À ce stade, cette version du notebook **matérialise le besoin** et refuse
de pré-affecter le gate à `True`.


In [14]:

CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED = False
CONNECTION_FRECHET_JET_CONTRACTED_IN_Y = False
Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED = False

if (
    CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED
    and
    CONNECTION_FRECHET_JET_CONTRACTED_IN_Y
):
    Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED = True

assert not Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED

print(
    "CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED =",
    CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED
)
print(
    "CONNECTION_FRECHET_JET_CONTRACTED_IN_Y =",
    CONNECTION_FRECHET_JET_CONTRACTED_IN_Y
)
print(
    "Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED =",
    Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED
)


CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED = False
CONNECTION_FRECHET_JET_CONTRACTED_IN_Y = False
Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED = False



# 14 — Conséquence pour l'inverse distributionnel

La structure abstraite :

\[
C=
\begin{pmatrix}
0&0&0&-\Delta\\
0&0&\Delta&\mathsf X\\
0&-\Delta&0&\mathsf Y\\
\Delta&-\mathsf X^\dagger&-\mathsf Y^\dagger&0
\end{pmatrix}
\]

reste algébriquement correcte sur la branche où l'opérateur
multiplicatif \(\Delta_{\rm FF}\) est inversible.

Mais le mot **véritable inverse distributionnel matérialisé** est réservé
au moment où :

\[
\mathsf X,\quad
\mathsf X^\dagger,\quad
\mathsf Y,\quad
\mathsf Y^\dagger
\]

sont tous explicitement construits sur le même domaine.

Donc le bloc inférieur droit nul de l'inverse abstrait reste un résultat
structurel, pas encore une clôture fonctionnelle finale.


In [15]:

DISTRIBUTIONAL_DIRAC_INVERSE_STRUCTURAL = (
    X_DISTRIBUTIONAL_KERNEL_MATERIALIZED
)

DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED = (
    X_DISTRIBUTIONAL_KERNEL_MATERIALIZED
    and
    X_ADJOINT_FORMALIZED
    and
    Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED
)

assert DISTRIBUTIONAL_DIRAC_INVERSE_STRUCTURAL
assert not DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED

print(
    "DISTRIBUTIONAL_DIRAC_INVERSE_STRUCTURAL =",
    DISTRIBUTIONAL_DIRAC_INVERSE_STRUCTURAL
)
print(
    "DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED =",
    DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED
)


DISTRIBUTIONAL_DIRAC_INVERSE_STRUCTURAL = True
DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED = False



# 15 — Verdict strict de `.1.6.2.1.1`

Cette étape est volontairement asymétrique :

- \(\mathcal A_{\rm FF}\) est construit depuis le vrai \(Q,J_0,U_0\) ;
- \(\rho_{\rm FF}\) est donc réellement matérialisé ;
- \(\mathsf X\) est matérialisé comme opérateur
  \[
  X_0+X_1^iD_i;
  \]
- \(\mathsf Y\) exige encore le jet de Fréchet de courbure et de connexion
  contracté avec \(\rho_{\rm FF}\).

Par conséquent **aucun PASS Dirac final n'est pré-déclaré**.

Le résultat canonique reste :

\[
\boxed{
R_{HH}^{\rm can}=0
\quad\texttt{STRONG\_ZERO}
}
\]

et le Dirac reste :

\[
\boxed{
R_{HH}^{D}\approx0
\quad\texttt{CONDITIONAL/STRUCTURAL}
}
\]

jusqu'à matérialisation de \(\mathsf Y\).


In [16]:

GATES = {
    "Jacobi_independent_ansatz_used":
        JACOBI_INDEPENDENT_ANSATZ_USED,

    "Jacobi_Hessian_coefficient_forced_zero":
        JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO,

    "Jacobi_gradient_coefficient_forced_Dchi":
        JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI,

    "QJU_same_full_field_reconstructed":
        QJU_SAME_FULL_FIELD_RECONSTRUCTED,

    "spatial_jet_operator_ready":
        SPATIAL_JET_OPERATOR_READY,

    "scalar_vector_functional_jets_explicit":
        SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT,

    "metric_algebraic_jets_explicit":
        METRIC_ALGEBRAIC_JETS_EXPLICIT,

    "connection_local_blocks_explicit":
        CONNECTION_LOCAL_BLOCKS_EXPLICIT,

    "A_FF_materialized_from_QJU":
        A_FF_MATERIALIZED_FROM_QJU,

    "rho_FF_materialized_from_QJU":
        RHO_FF_MATERIALIZED_FROM_QJU,

    "X_distributional_kernel_materialized":
        X_DISTRIBUTIONAL_KERNEL_MATERIALIZED,

    "X_adjoint_formalized":
        X_ADJOINT_FORMALIZED,

    "curvature_Frechet_second_variation_contracted":
        CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED,

    "connection_Frechet_jet_contracted_in_Y":
        CONNECTION_FRECHET_JET_CONTRACTED_IN_Y,

    "Y_distributional_kernel_materialized":
        Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED,

    "distributional_Dirac_inverse_structural":
        DISTRIBUTIONAL_DIRAC_INVERSE_STRUCTURAL,

    "distributional_Dirac_inverse_fully_materialized":
        DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED,

    "RHH_physical_classified":
        False,

    "dispersion_ready":
        False,
}

for k,vv in GATES.items():
    print(k,":",vv)

if (
    A_FF_MATERIALIZED_FROM_QJU
    and
    RHO_FF_MATERIALIZED_FROM_QJU
    and
    X_DISTRIBUTIONAL_KERNEL_MATERIALIZED
    and
    not Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED
):
    FINAL_STATUS = (
        "PARTIAL-PASS-EXPLICIT-AFF-RHOFF-X-DISTRIBUTIONAL-KERNEL_"
        "BLOCKED-Y-CURVATURE-AND-CONNECTION-FRECHET-MATERIALIZATION"
    )
else:
    FINAL_STATUS = (
        "BLOCKED-STRICT-DIRAC-MATERIALIZATION-INCOMPLETE"
    )

R_HH_OPERATIONAL_STATUS = (
    "CANONICAL-STRONG-ZERO_"
    "DIRAC-CONDITIONAL-STRUCTURAL"
)

NEXT_STATUS = (
    "ONLY-Y-DISTRIBUTIONAL-FRECHET-KERNEL-"
    "CURVATURE-CONNECTION-CONTRACTION-AUTHORIZED"
)

print("FINAL_STATUS =",FINAL_STATUS)
print("R_HH_OPERATIONAL_STATUS =",R_HH_OPERATIONAL_STATUS)
print("NEXT_STATUS =",NEXT_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


Jacobi_independent_ansatz_used : True
Jacobi_Hessian_coefficient_forced_zero : True
Jacobi_gradient_coefficient_forced_Dchi : True
QJU_same_full_field_reconstructed : True
spatial_jet_operator_ready : True
scalar_vector_functional_jets_explicit : True
metric_algebraic_jets_explicit : True
connection_local_blocks_explicit : True
A_FF_materialized_from_QJU : True
rho_FF_materialized_from_QJU : True
X_distributional_kernel_materialized : True
X_adjoint_formalized : True
curvature_Frechet_second_variation_contracted : False
connection_Frechet_jet_contracted_in_Y : False
Y_distributional_kernel_materialized : False
distributional_Dirac_inverse_structural : True
distributional_Dirac_inverse_fully_materialized : False
RHH_physical_classified : False
dispersion_ready : False
FINAL_STATUS = PARTIAL-PASS-EXPLICIT-AFF-RHOFF-X-DISTRIBUTIONAL-KERNEL_BLOCKED-Y-CURVATURE-AND-CONNECTION-FRECHET-MATERIALIZATION
R_HH_OPERATIONAL_STATUS = CANONICAL-STRONG-ZERO_DIRAC-CONDITIONAL-STRUCTURAL
NEXT_STATUS = O


# 16 — Export machine-readable


In [17]:

artifact = {
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.6.2.1.1",

    "frozen_input":
        {
            "canonical_RHH":
                "STRONG_ZERO",

            "Dirac_RHH":
                "CONDITIONAL_STRUCTURAL_WEAK_ZERO_GENERIC_BRANCH",
        },

    "materialized":
        {
            "A_FF":
                A_FF_MATERIALIZED_FROM_QJU,

            "rho_FF":
                RHO_FF_MATERIALIZED_FROM_QJU,

            "X_kernel":
                X_DISTRIBUTIONAL_KERNEL_MATERIALIZED,

            "Y_kernel":
                Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED,
        },

    "X_kernel":
        {
            "delta_coefficient":
                str(Xker0),

            "Ddelta_coefficients":
                [str(z) for z in Xker1],

            "formal_adjoint":
                "a - D_i b^i - b^i D_i",

            "domain":
                BOUNDARY_DOMAIN,
        },

    "rho_dependencies":
        rho_dependencies,

    "gates":
        GATES,

    "final_status":
        FINAL_STATUS,

    "R_HH_operational_status":
        R_HH_OPERATIONAL_STATUS,

    "next_status":
        NEXT_STATUS,

    "dispersion_ready":
        False,
}

export_dir = (
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path.cwd()/"gvh_exports"
)

export_dir.mkdir(parents=True,exist_ok=True)

artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.1.6.2.1.1_"
    "strict_Dirac_materialization.json"
)

artifact_path.write_text(
    json.dumps(artifact,indent=2),
    encoding="utf-8"
)

print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.1.6.2.1.1_strict_Dirac_materialization.json



# Conclusion

`.1.6.2.1.1` ne doit pas être utilisé pour forcer un verdict.

Il corrige d'abord le raisonnement Jacobi en démontrant, à partir d'un
ansatz libre :

\[
\beta^{ij}=0,
\qquad
\alpha^i=D^i\chi.
\]

Il calcule ensuite :

\[
\boxed{
\mathcal A_{\rm FF}
=
\{\psi,H_0[1]\}
}
\]

depuis les vrais jets fonctionnels de \(Q,J_0,U_0\), puis :

\[
\boxed{
\rho_{\rm FF}
=
\mathcal A_{\rm FF}
+
\lambda\Delta_{\rm FF}.
}
\]

Le premier noyau distributionnel est construit sous la forme :

\[
\boxed{
\mathsf X
=
X_0+X_1^iD_i.
}
\]

Le second noyau \(\mathsf Y\) n'est autorisé à devenir `True` qu'après
contraction explicite du jet de Fréchet de la courbure spatiale et du
jet de connexion.

Tant que ce dernier calcul n'est pas matérialisé :

\[
\boxed{
R_{HH}^{D}\approx0
\quad
\texttt{CONDITIONAL/STRUCTURAL}
}
\]

reste le statut scientifique prudent, et :

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
